# Predictive Models: Google Merchandise Store (GA4)
**ITC C612: Analytics Techniques & Tools** · Shellah Bea Labanza · 4th Year BSIT-BA – 401BA

This notebook carries out **Section 5 (Model Evaluation Plan)** of the Problem Statement. Each descriptive finding is turned into a predictive task, and every task follows the same steps that the plan requires:

| Step | What it means here |
|---|---|
| Target variable | The exact value the model predicts |
| Feature selection rationale | Which columns are used, which are left out, and why |
| Train/test split | How the data is divided (time-based for forecasting) |
| Baseline | A simple rule that every model must beat |
| Models compared | The models listed in Table 1 of the Problem Statement |
| Metrics | RMSE, MAE, R² for numbers; accuracy, precision, recall, F1, AUC for classes; coverage for ranges |
| Interpretation | Which features matter most |
| Overfitting | Training score vs. test score |

**The four tasks**

| Problem | Task | Model(s) |
|---|---|---|
| #1 Frequency | Will a visitor buy? | Logistic regression vs. gradient boosting |
| #2 Central tendency | Order value; daily traffic | Ridge regression on log revenue; Holt-Winters |
| #3 Dispersion | A range around each prediction | Quantile regression; Holt-Winters forecast intervals |
| #4 Position | High-value orders; quiet days | Decision tree vs. logistic regression |

**How to use this notebook**
1. Open it in Google Colab.
2. Choose **Runtime → Run all**. When asked, upload `BigQuery_4M_100k_cleaned.csv`.
3. At the end, the notebook downloads **`predictions.json`**. Copy that file into the `ecommerce-gms-main/` folder, next to `data.json`, and the dashboard's **Predictions** panel will show the results.

> Every random step uses `RANDOM_STATE = 42`, so running the notebook again gives the same results.

## 0. Setup

All the libraries below come pre-installed in Google Colab, so no `pip install` is needed.

- **pandas / numpy**: loading and reshaping the data.
- **scikit-learn**: logistic regression, gradient boosting, Ridge, quantile regression, decision tree, splits and metrics.
- **statsmodels**: Holt-Winters (exponential smoothing), which also gives forecast intervals.
- **matplotlib**: a few charts to check the results by eye.

In [ ]:
import json
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     GridSearchCV, cross_validate, cross_val_predict)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeCV, QuantileRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, mean_absolute_error, mean_squared_error, r2_score)
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

warnings.filterwarnings("ignore")          # keeps the output readable; remove to see all warnings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

RANDOM_STATE = 42

## 1. Load the cleaned data

The notebook looks for `BigQuery_4M_100k_cleaned.csv` in the current folder (or the folder above it). In Colab, if the file is not there yet, an upload button appears.

*Tip:* if the file is in Google Drive, you can mount Drive instead (`from google.colab import drive; drive.mount('/content/drive')`) and set `CSV_PATH` to its Drive path.

In [ ]:
CSV_NAME = "BigQuery_4M_100k_cleaned.csv"

try:
    from google.colab import files as colab_files   # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CSV_PATH = next((p for p in [CSV_NAME, os.path.join("..", CSV_NAME)] if os.path.exists(p)), None)
if CSV_PATH is None and IN_COLAB:
    print(f"Please upload {CSV_NAME}")
    uploaded = colab_files.upload()
    CSV_PATH = CSV_NAME
if CSV_PATH is None:
    raise FileNotFoundError(f"Put {CSV_NAME} next to this notebook and run again.")

# user_pseudo_id is an ID, not a number, so it is read as text
df = pd.read_csv(CSV_PATH, low_memory=False, parse_dates=["event_time"],
                 dtype={"user_pseudo_id": str})
df["event_date"] = pd.to_datetime(df["event_date"])
print(f"Loaded {CSV_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 2. Data check

Before modeling, confirm that the file matches the numbers in the Problem Statement (Section 4). If these numbers differ, the wrong file was loaded.

| Check | Expected (Problem Statement) |
|---|---|
| Rows / visitors / days | 100,000 / 42,610 / 92 (Nov 1, 2020 – Jan 31, 2021) |
| Purchase events | 133 (0.13% of events) |
| Orders with real revenue | 123 (mean $71.52, median $50) |
| Daily events | mean ≈ 1,087, median ≈ 1,078.5 |

**About the 10 orders without revenue.** The cleaned CSV filled 10 missing purchase revenues with the $50 median and marked them in `purchase_missing_revenue`. A filled-in value is not a real order value, so those 10 orders are left out of every revenue model. That is why the order models use 123 orders, not 133.

**Columns not used.** `user_id` and `user_id_usable` (almost always empty), `platform` and `stream_id` (one value for every row), and the `*_is_placeholder` / `*_missing` / `*_imputed` flags (data-quality markers, not visitor behavior).

In [ ]:
DROP_COLS = ["user_id", "user_id_usable", "platform", "stream_id"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

is_purchase = df["event_name"].eq("purchase")
real_revenue = is_purchase & ~df["purchase_missing_revenue"].astype(bool)
daily_events = df.groupby("event_date").size()

check = pd.DataFrame({
    "value": [
        f"{len(df):,}",
        f"{df['user_pseudo_id'].nunique():,}",
        f"{daily_events.size} ({daily_events.index.min():%b %d, %Y} – {daily_events.index.max():%b %d, %Y})",
        f"{is_purchase.sum()} ({is_purchase.mean():.2%})",
        f"{real_revenue.sum()} (mean ${df.loc[real_revenue, 'purchase_revenue'].mean():.2f}, "
        f"median ${df.loc[real_revenue, 'purchase_revenue'].median():.2f})",
        f"mean {daily_events.mean():,.1f}, median {daily_events.median():,.1f}",
    ]},
    index=["Rows", "Visitors", "Days", "Purchase events", "Orders with real revenue", "Daily events"])
check

## 3. Shared helpers

Small functions reused by every task so the metrics are calculated the same way everywhere.

- `clf_scores`: accuracy, precision, recall, F1 and AUC for a yes/no prediction. `zero_division=0` means a model that never predicts "yes" gets a precision of 0 instead of an error.
- `reg_scores`: MAE, RMSE and R² for a number prediction.
- `coverage`: the share of actual values that fall inside a predicted range.

In [ ]:
def clf_scores(y_true, y_pred, y_prob=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    out["auc"] = (roc_auc_score(y_true, y_prob)
                  if y_prob is not None and len(np.unique(y_true)) > 1 else np.nan)
    return out


def reg_scores(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


def coverage(y_true, lower, upper):
    y_true, lower, upper = map(np.asarray, (y_true, lower, upper))
    return float(np.mean((y_true >= lower) & (y_true <= upper)))


def top_categories(series, n, other="Other"):
    # Keeps the n most common values and groups the rest, so rare values
    # do not create columns with only one or two rows.
    keep = series.value_counts().index[:n]
    return series.where(series.isin(keep), other)


RESULTS = {}   # everything that goes into predictions.json is collected here

## 4. Building the three modeling tables

The raw file has one row per **event**. Each task needs a different unit, so three tables are built:

| Table | One row per | Used by |
|---|---|---|
| `visitors` | visitor (`user_pseudo_id`) | Problem #1 |
| `orders` | purchase with real revenue | Problems #2, #3, #4 (orders) |
| `days` | calendar day | Problems #2, #3, #4 (days) |

### 4.1 Visitor table (Problem #1)

**Features kept** (Section 5.1 of the plan):
- *Browsing counts*: page views, item views, promotion views, scrolls and sessions. They show how engaged a visitor is.
- *Device*: the mix is 58.6% desktop, and device can change how people shop.
- *Traffic channel* (`channel_group`): reflects intent (someone who searched vs. someone who clicked an ad).
- *Country*: the 10 most common countries, the rest grouped as "Other".

Device, channel and country are taken from the visitor's **first event**, i.e. how they arrived.

**Features left out on purpose (data leakage).** `begin_checkout`, `add_shipping_info` and `add_payment_info` happen just before a purchase. A model that sees them would "predict" buyers only because they were already paying, which is useless for targeting offers in advance. `purchase` itself is the target, so it is also excluded.

In [ ]:
BROWSE_EVENTS = {
    "page_view": "n_page_view",
    "view_item": "n_view_item",
    "view_promotion": "n_view_promotion",
    "scroll": "n_scroll",
    "session_start": "n_session_start",
}
LEAKY_EVENTS = ["begin_checkout", "add_shipping_info", "add_payment_info", "purchase"]

event_counts = pd.crosstab(df["user_pseudo_id"], df["event_name"])
first_event = (df.sort_values("event_time")
                 .groupby("user_pseudo_id")[["device_category", "channel_group", "country"]]
                 .first())
first_event["country"] = top_categories(first_event["country"], 10)

visitors = event_counts.reindex(columns=list(BROWSE_EVENTS), fill_value=0).rename(columns=BROWSE_EVENTS)
visitors = visitors.join(first_event)
visitors["buyer"] = (event_counts["purchase"] > 0).astype(int)

print(f"Visitors: {len(visitors):,}   Buyers: {visitors['buyer'].sum()} ({visitors['buyer'].mean():.2%})")
print("Leaky events excluded from the features:", LEAKY_EVENTS)
visitors.head()

### 4.2 Order table (Problems #2, #3, #4)

One row per purchase with real revenue (123 orders).

**Features** (Section 5.2): device and channel (the type of shopper), country (the 5 most common, the rest "Other"), and day of week and month (holiday spending in Nov–Dec).

**Why so few country groups?** With only 123 orders, every extra category is a new column with very few rows, which makes overfitting easy.

**Target.** `log_revenue = log(revenue)`. Order values are skewed (mean $71.52 > median $50), and the log makes the distribution more even so a few large orders do not dominate the model. Predictions are converted back to dollars before scoring.

In [ ]:
orders = df.loc[real_revenue, ["event_date", "device_category", "channel_group",
                               "country", "purchase_revenue"]].copy()
orders["country"] = top_categories(orders["country"], 5)
orders["day_of_week"] = orders["event_date"].dt.day_name()
orders["month"] = orders["event_date"].dt.strftime("%b")
orders["log_revenue"] = np.log(orders["purchase_revenue"])
orders = orders.reset_index(drop=True)

ORDER_FEATURES = ["device_category", "channel_group", "country", "day_of_week", "month"]
# One-hot encoding: each category becomes a 0/1 column. drop_first=True removes one
# category per feature (the "reference"), so each coefficient reads as
# "compared with the reference category".
X_orders = pd.get_dummies(orders[ORDER_FEATURES], drop_first=True, dtype=float)

print(f"Orders: {len(orders)}   Feature columns after one-hot encoding: {X_orders.shape[1]}")
orders.head()

### 4.3 Daily table (Problems #2, #3, #4)

One row per day (92 days) with the total number of events.

Extra columns for Problem #4 (Section 5.4):
- `day_of_week`: quiet days follow the weekly pattern.
- `holiday`: 1 on Thanksgiving, Black Friday, Cyber Monday, Christmas Eve/Day, New Year's Eve/Day.
- `lag7`: events on the same weekday one week earlier. Only past values are used, so the model never sees the future.

In [ ]:
HOLIDAYS = pd.to_datetime(["2020-11-26", "2020-11-27", "2020-11-30",
                           "2020-12-24", "2020-12-25", "2020-12-31", "2021-01-01"])

days = daily_events.rename("events").asfreq("D").to_frame()
days["day_of_week"] = days.index.day_name()
days["holiday"] = days.index.isin(HOLIDAYS).astype(int)
days["lag7"] = days["events"].shift(7)

N_TEST_DAYS = 14                         # the two-week forecast in Section 4.2
N_TRAIN_DAYS = len(days) - N_TEST_DAYS   # 78
train_days = days.iloc[:N_TRAIN_DAYS]
test_days = days.iloc[N_TRAIN_DAYS:]
print(f"Days: {len(days)}   Train: {train_days.index[0]:%b %d} – {train_days.index[-1]:%b %d} ({len(train_days)})"
      f"   Test: {test_days.index[0]:%b %d} – {test_days.index[-1]:%b %d} ({len(test_days)})")
days.head(10)

---
## 5. Problem #1 (Frequency): Who will buy?

| Item | Plan (Section 5.1) |
|---|---|
| Target | `buyer`: 1 if the visitor has at least one purchase, 0 if not (≈0.3% are 1) |
| Split | 80/20, **stratified** so both sets keep the 0.3% buyer rate |
| Baseline | Predict "no buyer" for everyone |
| Models | Logistic regression (main) vs. gradient boosting (check) |
| Metrics | Precision, recall, F1, AUC. **Recall and AUC are the main ones** |

**Why stratify?** With only ~127 buyers, a plain random split could put very few buyers in the test set by chance. Stratifying keeps the same buyer share in both sets.

### 5.1 Split and baseline
The baseline shows why accuracy is a misleading metric here: predicting "no buyer" for everyone is about 99.7% accurate, yet it finds **zero** buyers (recall = 0).

In [ ]:
P1_FEATURES = list(BROWSE_EVENTS.values()) + ["device_category", "channel_group", "country"]
X1 = pd.get_dummies(visitors[P1_FEATURES], drop_first=True, dtype=float)
y1 = visitors["buyer"]

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, stratify=y1, random_state=RANDOM_STATE)
print(f"Train: {len(y1_train):,} visitors ({y1_train.sum()} buyers)   "
      f"Test: {len(y1_test):,} visitors ({y1_test.sum()} buyers)")

base1 = DummyClassifier(strategy="most_frequent").fit(X1_train, y1_train)
p1_scores = {"Baseline (no buyer)": clf_scores(y1_test, base1.predict(X1_test),
                                               base1.predict_proba(X1_test)[:, 1])}
pd.DataFrame(p1_scores).T

### 5.2 Model A: Logistic regression (main model)

Logistic regression gives each visitor a probability of buying. It is easy to explain because every coefficient can be turned into an **odds ratio**.

- `class_weight="balanced"`: buyers are rare, so each buyer counts more during training. Without it, the model would learn to predict "no" for everyone, just like the baseline.
- **Regularization (`C`)**: a smaller `C` keeps the coefficients smaller, which protects against memorizing the ~100 training buyers. The best `C` is chosen by **5-fold stratified cross-validation on the training set only**, using AUC.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_search = GridSearchCV(
    LogisticRegression(class_weight="balanced", max_iter=5000),
    param_grid={"C": [0.01, 0.1, 1, 10]},
    scoring="roc_auc", cv=cv5)
lr_search.fit(X1_train, y1_train)
lr1 = lr_search.best_estimator_
print("Best C:", lr_search.best_params_["C"], f"  (cross-validated AUC {lr_search.best_score_:.3f})")

lr1_prob_test = lr1.predict_proba(X1_test)[:, 1]
p1_scores["Logistic regression"] = clf_scores(y1_test, lr1.predict(X1_test), lr1_prob_test)

### 5.3 Model B: Gradient boosting (check)

Gradient boosting builds many small decision trees, each one fixing the mistakes of the previous ones. It can find patterns that logistic regression misses (for example, "many item views **and** a desktop device"). If it scores much higher, the simple model is missing something; if it scores about the same, the simpler logistic regression is kept.

Overfitting controls (Section 5.1):
- `max_depth=2`: each tree is very small.
- **Early stopping** (`n_iter_no_change=20`): training stops when the score on an internal validation slice stops improving.
- `subsample=0.8`: each tree sees a random 80% of the data.
- Balanced sample weights play the same role as `class_weight="balanced"` above.

In [ ]:
gb1 = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=2, subsample=0.8,
    n_iter_no_change=20, validation_fraction=0.2, random_state=RANDOM_STATE)
gb1.fit(X1_train, y1_train, sample_weight=compute_sample_weight("balanced", y1_train))
print("Trees used after early stopping:", gb1.n_estimators_)

gb1_prob_test = gb1.predict_proba(X1_test)[:, 1]
p1_scores["Gradient boosting"] = clf_scores(y1_test, gb1.predict(X1_test), gb1_prob_test)

p1_table = pd.DataFrame(p1_scores).T
p1_table

**How to read this table**
- **Recall**: of the real buyers in the test set, the share the model flagged. The baseline's recall is 0.
- **Precision**: of the visitors flagged as buyers, the share who actually bought. It will be low because buyers are so rare; that is expected and acceptable when the goal is to rank visitors for offers.
- **AUC**: the chance that a random buyer gets a higher score than a random non-buyer. 0.5 is guessing (the baseline); 1.0 is perfect.
- **Accuracy**: shown only to prove the point from 5.1. It is not used to choose the model.

### 5.4 Interpretation of important features

- **Logistic regression → odds ratios.** `exp(coefficient)` is the odds ratio. An odds ratio of 1.25 for `n_view_item` means *each extra item viewed raises the odds of buying by 25%*. Below 1 lowers the odds. For device/channel/country, the ratio compares with the reference category that was dropped.
- **Gradient boosting → feature importance.** How much each feature helped the trees split buyers from non-buyers. It shows strength but not direction.

If both models put the same features at the top, the finding is more trustworthy.

In [ ]:
odds = (pd.DataFrame({"feature": X1.columns, "coefficient": lr1.coef_[0]})
          .assign(odds_ratio=lambda d: np.exp(d["coefficient"]))
          .assign(abs_effect=lambda d: d["coefficient"].abs())
          .sort_values("abs_effect", ascending=False)
          .drop(columns="abs_effect"))
gb_imp = (pd.DataFrame({"feature": X1.columns, "importance": gb1.feature_importances_})
            .sort_values("importance", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
odds.head(10).iloc[::-1].plot.barh(x="feature", y="odds_ratio", ax=axes[0], legend=False, color="#1f3b73")
axes[0].axvline(1, color="grey", ls="--"); axes[0].set_title("Logistic regression: top 10 odds ratios")
gb_imp.head(10).iloc[::-1].plot.barh(x="feature", y="importance", ax=axes[1], legend=False, color="#2aa1b3")
axes[1].set_title("Gradient boosting: top 10 feature importances")
plt.tight_layout(); plt.show()

display(odds.head(10)); display(gb_imp.head(10))

### 5.5 Overfitting check

Compare AUC on the training set and the test set. A training AUC far above the test AUC (for example 0.98 vs. 0.75) means the model memorized the training buyers instead of learning patterns that carry over to new visitors.

In [ ]:
p1_overfit = pd.DataFrame({
    "train_auc": [roc_auc_score(y1_train, lr1.predict_proba(X1_train)[:, 1]),
                  roc_auc_score(y1_train, gb1.predict_proba(X1_train)[:, 1])],
    "test_auc": [p1_scores["Logistic regression"]["auc"], p1_scores["Gradient boosting"]["auc"]],
}, index=["Logistic regression", "Gradient boosting"])
p1_overfit["gap"] = p1_overfit["train_auc"] - p1_overfit["test_auc"]
p1_overfit

### 5.6 Output: ranked list of visitors

The Problem Statement's output for #1 is *a ranked list of visitors by buying likelihood*. The list below ranks the **test-set** visitors (people the model never saw) by the logistic regression score, so it shows honestly how well the ranking works. `share_of_buyers_in_top_1pct` tells you what share of all test buyers sit in the top 1% of the list.

In [ ]:
ranked = (pd.DataFrame({"visitor": X1_test.index, "probability": lr1_prob_test,
                        "actual_buyer": y1_test.values})
            .sort_values("probability", ascending=False)
            .reset_index(drop=True))
top_n = max(1, int(len(ranked) * 0.01))
share_top1 = ranked.head(top_n)["actual_buyer"].sum() / max(1, ranked["actual_buyer"].sum())
print(f"Top 1% of test visitors = {top_n} people; they contain {share_top1:.0%} of all test buyers.")
ranked.head(20)

In [ ]:
RESULTS["p1"] = {
    "n_visitors": int(len(visitors)), "n_buyers": int(visitors["buyer"].sum()),
    "best_C": lr_search.best_params_["C"], "gb_trees": int(gb1.n_estimators_),
    "metrics": p1_table.reset_index(names="model").to_dict("records"),
    "overfit": p1_overfit.reset_index(names="model").to_dict("records"),
    "odds_ratios": odds.head(10).to_dict("records"),
    "gb_importance": gb_imp.head(10).to_dict("records"),
    "top_visitors": ranked.head(20).to_dict("records"),
    "share_buyers_top1pct": share_top1,
}

---
## 6. Problem #2 (Central tendency): Typical order and typical day

### 6A. Order value: Ridge regression

| Item | Plan (Section 5.2) |
|---|---|
| Target | Revenue per order (123 orders), modeled as **log revenue** |
| Split | Random 80/20 **plus 5-fold cross-validation** on the training set |
| Baseline | Predict the median order (≈ $50) for every order |
| Model | Ridge regression |
| Metrics | MAE, RMSE, R² (in dollars) |

**Why random and not time-based?** Orders are not a time series; one order does not depend on the previous one. **Why cross-validation too?** With only ~25 test orders, one split can be lucky or unlucky. 5-fold CV tests the model on 5 different slices of the training data and gives a steadier estimate.

**Why Ridge?** Ridge is linear regression with a penalty (`alpha`) that shrinks coefficients toward zero. With many one-hot columns and few orders, that penalty stops the model from fitting noise. `RidgeCV` picks `alpha` by cross-validation on the training set.

In [ ]:
Xo_train, Xo_test, yo_train, yo_test = train_test_split(
    X_orders, orders["log_revenue"], test_size=0.2, random_state=RANDOM_STATE)
rev_train, rev_test = np.exp(yo_train), np.exp(yo_test)          # dollars, for scoring
print(f"Train orders: {len(yo_train)}   Test orders: {len(yo_test)}")

# Baseline: the median of the TRAINING orders (close to the $50 in Table 4)
median_train = float(np.median(rev_train))
print(f"Baseline prediction (training median): ${median_train:.2f}")

ALPHAS = np.logspace(-2, 3, 30)
ridge = RidgeCV(alphas=ALPHAS, cv=5).fit(Xo_train, yo_train)
print(f"Ridge alpha chosen by cross-validation: {ridge.alpha_:.3f}")

# Predictions are made on the log scale and converted back with exp()
ridge_pred_test = np.exp(ridge.predict(Xo_test))

p2o_scores = {
    "Baseline (training median)": reg_scores(rev_test, np.full(len(rev_test), median_train)),
    "Ridge regression": reg_scores(rev_test, ridge_pred_test),
}
pd.DataFrame(p2o_scores).T

**5-fold cross-validation on the training orders.** The same comparison repeated on 5 slices of the training set. In each fold, the baseline median and Ridge's `alpha` are recalculated from that fold's training part only.

In [ ]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for fold, (tr, va) in enumerate(kf5.split(Xo_train), start=1):
    m = RidgeCV(alphas=ALPHAS).fit(Xo_train.iloc[tr], yo_train.iloc[tr])
    actual = np.exp(yo_train.iloc[va])
    base_pred = np.full(len(va), np.median(np.exp(yo_train.iloc[tr])))
    cv_rows.append({"fold": fold,
                    "baseline_MAE": mean_absolute_error(actual, base_pred),
                    "ridge_MAE": mean_absolute_error(actual, np.exp(m.predict(Xo_train.iloc[va])))})
p2o_cv = pd.DataFrame(cv_rows)
display(p2o_cv)
print(f"Mean CV MAE  baseline ${p2o_cv['baseline_MAE'].mean():.2f}   ridge ${p2o_cv['ridge_MAE'].mean():.2f}")

**How to read these numbers**
- **MAE**: the average miss in dollars ("on average the prediction is off by $X").
- **RMSE**: like MAE but large misses count more; a few very large orders push it up.
- **R²**: the share of the variation in order value the model explains. 0 means no better than one fixed number; **a low or even negative R² is expected** (Section 5.2) because the data has no product details, and product choice is what really drives order value.

**Note on the CV result.** If the cross-validated `alpha` is large (in the hundreds), Ridge has shrunk almost every coefficient to zero and is predicting close to the median for everyone. That is the model saying the features carry very little information about order value, which matches the limitation in Section 6 (no product details).

**Interpretation of important features.** Because the target is log revenue, `exp(coefficient) − 1` is the **% change in order value** compared with the reference category. For example, +20% means orders from that group are about 20% larger, all else equal.

In [ ]:
ridge_effects = (pd.DataFrame({"feature": X_orders.columns, "coefficient": ridge.coef_})
                   .assign(pct_change_in_order_value=lambda d: (np.exp(d["coefficient"]) - 1) * 100)
                   .assign(abs_effect=lambda d: d["coefficient"].abs())
                   .sort_values("abs_effect", ascending=False).drop(columns="abs_effect"))
ridge_effects.head(10)

**Overfitting check.** Compare training and test RMSE. Ridge's penalty already limits overfitting; a test RMSE far above the training RMSE would mean the penalty is too weak.

In [ ]:
p2o_overfit = pd.DataFrame({
    "train_RMSE": [reg_scores(rev_train, np.exp(ridge.predict(Xo_train)))["RMSE"]],
    "test_RMSE": [p2o_scores["Ridge regression"]["RMSE"]],
}, index=["Ridge regression"])
p2o_overfit["gap"] = p2o_overfit["test_RMSE"] - p2o_overfit["train_RMSE"]
p2o_overfit

### 6B. Daily traffic: Holt-Winters

| Item | Plan (Section 5.2) |
|---|---|
| Target | Total events per day (92 days) |
| Split | **Time-based**: train on the first 78 days, test on the last 14. Days are never shuffled |
| Baseline | Seasonal naive: each day = the same weekday one week earlier |
| Model | Holt-Winters |
| Metrics | MAE, RMSE (events per day) |

**Why time-based?** A forecast is always made from the past toward the future. Shuffling days would let the model "peek" at days after the ones it predicts, making the scores unrealistically good.

**Holt-Winters** smooths three parts of the series: the **level** (current typical volume), the **trend** (rising or falling) and a **7-day seasonal** cycle (the weekly pattern seen in Section 4.2). It is fitted here as a state-space model (`ETSModel`) because that version also produces the forecast intervals needed for Problem #3. The trend is **damped**, so it flattens out over the 14 days instead of continuing the holiday rise or the January drop in a straight line.

**Seasonal naive for 14 days ahead.** On the forecast day (end of day 78), only the last real week is known. So the baseline repeats the last training week twice.

In [ ]:
y_train_d, y_test_d = train_days["events"].astype(float), test_days["events"].astype(float)

hw = ETSModel(y_train_d, error="add", trend="add", damped_trend=True,
              seasonal="add", seasonal_periods=7).fit(disp=False)

hw_pred = hw.get_prediction(start=y_test_d.index[0], end=y_test_d.index[-1]).summary_frame(alpha=0.20)
hw_forecast = hw_pred["mean"]

naive_forecast = pd.Series(np.tile(y_train_d.iloc[-7:].values, 2)[:N_TEST_DAYS], index=y_test_d.index)

p2d_scores = {
    "Baseline (seasonal naive)": reg_scores(y_test_d, naive_forecast),
    "Holt-Winters": reg_scores(y_test_d, hw_forecast),
}
pd.DataFrame(p2d_scores).T[["MAE", "RMSE"]]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(days.index, days["events"], color="black", lw=1, label="Actual")
ax.plot(hw_forecast.index, hw_forecast, color="#1f3b73", lw=2, label="Holt-Winters forecast")
ax.plot(naive_forecast.index, naive_forecast, color="#c08a3e", lw=1.5, ls="--", label="Seasonal naive")
ax.axvline(y_test_d.index[0], color="grey", ls=":"); ax.text(y_test_d.index[0], ax.get_ylim()[1]*0.95, "  test period")
ax.set_title("Daily events: training period, test period and forecasts"); ax.legend()
plt.show()

**Interpretation.** Holt-Winters has no "features" in the usual sense; it learns from the series itself (Section 5.2). Its parts are read instead:
- **Weekly seasonal effects**: how many events above (+) or below (−) an average day each weekday usually is. (The model stores them relative to one weekday; they are re-centered on the weekly average so they are easier to read.)
- **Smoothing parameters**: `alpha` (how fast the level reacts), `beta` (trend), `gamma` (season), `phi` (trend damping, 1 = no damping).

In [ ]:
# The last 7 seasonal states, one per weekday, labelled with the weekday they belong to
last7 = hw.states["seasonal"].iloc[-7:]
weekday_effects = (pd.DataFrame({"weekday": last7.index.day_name(),
                                 "effect_vs_average_day": last7.values - last7.values.mean()})
                     .set_index("weekday")
                     .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]))
all_params = dict(zip(hw.model.param_names, np.asarray(hw.params)))
hw_params = {k: float(all_params[k]) for k in
             ("smoothing_level", "smoothing_trend", "smoothing_seasonal", "damping_trend")}
display(weekday_effects)
print("Smoothing parameters:", {k: round(v, 3) for k, v in hw_params.items()})

**Overfitting check.** Training error (how well the model fits the 78 days it learned from) vs. test error (the 14 days it never saw). The plan expects some gap because the training period holds the one holiday season, whose spikes will not repeat in late January.

In [ ]:
naive_train_err = (y_train_d - y_train_d.shift(7)).dropna()
p2d_overfit = pd.DataFrame({
    "train_MAE": [float(np.mean(np.abs(naive_train_err))),
                  mean_absolute_error(y_train_d, hw.fittedvalues)],
    "test_MAE": [p2d_scores["Baseline (seasonal naive)"]["MAE"], p2d_scores["Holt-Winters"]["MAE"]],
}, index=["Baseline (seasonal naive)", "Holt-Winters"])
p2d_overfit["gap"] = p2d_overfit["test_MAE"] - p2d_overfit["train_MAE"]
p2d_overfit

In [ ]:
RESULTS["p2"] = {
    "orders": {
        "n_train": int(len(yo_train)), "n_test": int(len(yo_test)),
        "baseline_value": median_train, "ridge_alpha": float(ridge.alpha_),
        "metrics": pd.DataFrame(p2o_scores).T.reset_index(names="model").to_dict("records"),
        "cv": p2o_cv.to_dict("records"),
        "effects": ridge_effects.head(10).to_dict("records"),
        "overfit": p2o_overfit.reset_index(names="model").to_dict("records"),
    },
    "traffic": {
        "metrics": pd.DataFrame(p2d_scores).T[["MAE", "RMSE"]].reset_index(names="model").to_dict("records"),
        "weekday_effects": weekday_effects.reset_index().to_dict("records"),
        "params": hw_params,
        "overfit": p2d_overfit.reset_index(names="model").to_dict("records"),
    },
}

---
## 7. Problem #3 (Dispersion): How certain are the predictions?

| Item | Plan (Section 5.3) |
|---|---|
| Target | Orders: low (P10), middle (P50) and high (P90) revenue. Days: lower and upper bands |
| Split | Same as Problem #2 (same orders, same 78/14 days), so results can be compared directly |
| Baseline | Orders: one fixed range for every order (training P10–P90, ≈ $15.20–$137 in Table 6). Days: a range around last week's same weekday |
| Models | Orders: quantile regression. Days: Holt-Winters forecast intervals |
| Metrics | MAE of the middle estimate, and **coverage** |

**Coverage** is the share of actual test values that fall inside the predicted range. A P10–P90 range should hold about **80%** of actual values:
- much **less** than 80% → the ranges are too narrow (over-confident);
- much **more** than 80% → the ranges are too wide to be useful.

### 7A. Order ranges: quantile regression

Ordinary regression predicts the average. **Quantile regression** predicts a chosen percentile instead, so three models give P10, P50 and P90 for each order. It is fitted on log revenue (the same target as Ridge); because percentiles keep their order under `log`/`exp`, converting back to dollars is exact. A small `alpha` penalty keeps the coefficients stable with so few orders.

In [ ]:
QUANTILES = {"P10": 0.10, "P50": 0.50, "P90": 0.90}
qr_models, qr_test, qr_train = {}, {}, {}
for name, q in QUANTILES.items():
    m = QuantileRegressor(quantile=q, alpha=0.01, solver="highs").fit(Xo_train, yo_train)
    qr_models[name] = m
    qr_test[name] = np.exp(m.predict(Xo_test))
    qr_train[name] = np.exp(m.predict(Xo_train))

# Make sure P10 <= P50 <= P90 for every order (separate models can occasionally cross)
for preds in (qr_test, qr_train):
    stacked = np.sort(np.vstack([preds["P10"], preds["P50"], preds["P90"]]), axis=0)
    preds["P10"], preds["P50"], preds["P90"] = stacked

# Baseline: one fixed range from the TRAINING orders only
b_p10, b_p50, b_p90 = np.percentile(rev_train, [10, 50, 90])
print(f"Baseline range (training P10–P90): ${b_p10:.2f} – ${b_p90:.2f}   (Table 6, all orders: $15.20 – $137.00)")

p3o_scores = {
    "Baseline (fixed range)": {
        "MAE_P50": mean_absolute_error(rev_test, np.full(len(rev_test), b_p50)),
        "coverage": coverage(rev_test, np.full(len(rev_test), b_p10), np.full(len(rev_test), b_p90)),
        "avg_width": b_p90 - b_p10},
    "Quantile regression": {
        "MAE_P50": mean_absolute_error(rev_test, qr_test["P50"]),
        "coverage": coverage(rev_test, qr_test["P10"], qr_test["P90"]),
        "avg_width": float(np.mean(qr_test["P90"] - qr_test["P10"]))},
}
pd.DataFrame(p3o_scores).T

**Interpretation of important features.** Comparing the P90 and P10 coefficients shows which features make the range **wider** (less certain). On the log scale, `P90 coefficient − P10 coefficient > 0` means that group's orders are more spread out.

In [ ]:
width_effects = (pd.DataFrame({"feature": X_orders.columns,
                               "coef_P10": qr_models["P10"].coef_,
                               "coef_P50": qr_models["P50"].coef_,
                               "coef_P90": qr_models["P90"].coef_})
                   .assign(widens_range_by=lambda d: d["coef_P90"] - d["coef_P10"])
                   .assign(abs_w=lambda d: d["widens_range_by"].abs())
                   .sort_values("abs_w", ascending=False).drop(columns="abs_w"))
width_effects.head(10)

**Reading coverage together with width.** A model can hit 80% coverage just by drawing a very wide range. So compare `avg_width` too: a range that is narrower than the baseline but covers almost as many orders is more useful, even if its coverage is a few points further from 80%.

**Overfitting check.** The P90 estimate rests on only about 12 orders (Section 5.3). If coverage is clearly higher on the training orders than on the test orders, the ranges fit the training data too closely and are too optimistic for new orders.

In [ ]:
p3o_overfit = pd.DataFrame({
    "train_coverage": [coverage(rev_train, qr_train["P10"], qr_train["P90"])],
    "test_coverage": [p3o_scores["Quantile regression"]["coverage"]],
}, index=["Quantile regression"])
p3o_overfit["gap"] = p3o_overfit["train_coverage"] - p3o_overfit["test_coverage"]
p3o_overfit

### 7B. Traffic ranges: Holt-Winters forecast intervals

The Holt-Winters model from 6B already gives an **80% interval** for each test day (`alpha=0.20` above). The baseline range takes last week's same weekday and adds the 10th and 90th percentile of how much a day differed from the same weekday one week earlier during training.

In [ ]:
err = naive_train_err
naive_lower = naive_forecast + np.percentile(err, 10)
naive_upper = naive_forecast + np.percentile(err, 90)

p3d_scores = {
    "Baseline (last week ± typical change)": {
        "MAE": mean_absolute_error(y_test_d, naive_forecast),
        "coverage": coverage(y_test_d, naive_lower, naive_upper),
        "avg_width": float(np.mean(naive_upper - naive_lower))},
    "Holt-Winters 80% interval": {
        "MAE": p2d_scores["Holt-Winters"]["MAE"],
        "coverage": coverage(y_test_d, hw_pred["pi_lower"], hw_pred["pi_upper"]),
        "avg_width": float(np.mean(hw_pred["pi_upper"] - hw_pred["pi_lower"]))},
}
display(pd.DataFrame(p3d_scores).T)

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(hw_pred.index, hw_pred["pi_lower"], hw_pred["pi_upper"], color="#2aa1b3", alpha=0.25, label="80% interval")
ax.plot(hw_pred.index, hw_pred["mean"], color="#1f3b73", label="Forecast")
ax.plot(y_test_d.index, y_test_d, "ko-", ms=4, label="Actual")
ax.set_title("Test period: Holt-Winters forecast with 80% interval"); ax.legend(); plt.show()

In [ ]:
# Overfitting check: in-sample coverage, using the training residual spread
resid_sd = float(np.std(hw.resid, ddof=1))
z80 = 1.2816   # 80% of a normal distribution lies within ±1.2816 standard deviations
train_cov_d = coverage(y_train_d, hw.fittedvalues - z80 * resid_sd, hw.fittedvalues + z80 * resid_sd)
p3d_overfit = pd.DataFrame({"train_coverage": [train_cov_d],
                            "test_coverage": [p3d_scores["Holt-Winters 80% interval"]["coverage"]]},
                           index=["Holt-Winters 80% interval"])
p3d_overfit["gap"] = p3d_overfit["train_coverage"] - p3d_overfit["test_coverage"]
p3d_overfit

### 7C. Forecast for the next two weeks (Feb 1–14, 2021)

The output promised in Sections 4.2 and 4.3: after the model has been evaluated, it is **refitted on all 92 days** and used to forecast the next 14 days, each with an 80% range. These are true forecasts; there are no actual values to compare them with yet.

In [ ]:
hw_full = ETSModel(days["events"].astype(float), error="add", trend="add", damped_trend=True,
                   seasonal="add", seasonal_periods=7).fit(disp=False)
future = hw_full.get_prediction(start=len(days), end=len(days) + N_TEST_DAYS - 1).summary_frame(alpha=0.20)
future.index = pd.date_range(days.index[-1] + pd.Timedelta(days=1), periods=N_TEST_DAYS, freq="D")
future.round(0)

In [ ]:
RESULTS["p3"] = {
    "orders": {
        "baseline_range": [float(b_p10), float(b_p90)],
        "metrics": pd.DataFrame(p3o_scores).T.reset_index(names="model").to_dict("records"),
        "width_effects": width_effects.head(10).to_dict("records"),
        "overfit": p3o_overfit.reset_index(names="model").to_dict("records"),
        "test_ranges": pd.DataFrame({"actual": rev_test.values, "P10": qr_test["P10"],
                                     "P50": qr_test["P50"], "P90": qr_test["P90"]}).to_dict("records"),
    },
    "traffic": {
        "metrics": pd.DataFrame(p3d_scores).T.reset_index(names="model").to_dict("records"),
        "overfit": p3d_overfit.reset_index(names="model").to_dict("records"),
    },
}
RESULTS["forecast"] = {
    "history_dates": [d.strftime("%Y-%m-%d") for d in days.index],
    "history_events": days["events"].astype(int).tolist(),
    "test_dates": [d.strftime("%Y-%m-%d") for d in hw_pred.index],
    "test_forecast": hw_pred["mean"].tolist(),
    "test_lower": hw_pred["pi_lower"].tolist(),
    "test_upper": hw_pred["pi_upper"].tolist(),
    "naive_forecast": naive_forecast.tolist(),
    "future_dates": [d.strftime("%Y-%m-%d") for d in future.index],
    "future_forecast": future["mean"].tolist(),
    "future_lower": future["pi_lower"].tolist(),
    "future_upper": future["pi_upper"].tolist(),
}

---
## 8. Problem #4 (Position): Which orders and days stand out?

| Item | Plan (Section 5.4) |
|---|---|
| Targets | High-value order: 1 if revenue ≥ P90. Quiet day: 1 if events < P25. **Cut-offs come from the training data only** |
| Split | Orders: stratified 80/20 + 5-fold CV. Days: time-based, last 14 days as test |
| Baseline | "Not high-value" for every order; "not quiet" for every day |
| Models | Decision tree vs. logistic regression |
| Metrics | Precision, recall, F1 (both); AUC for orders; accuracy for days shown next to the baseline. **Recall is the main one** |

**Why the cut-offs come from training data only.** If P90 were calculated on all orders, the test orders would help decide where the line is drawn, so the test would no longer be a clean "unseen data" check (data leakage).

### 8A. High-value orders

There are only about 12 high-value orders. To keep a few of them in the test set, the split is stratified on the "high-value" flag; then the P90 cut-off is **recalculated on the training orders alone** and the labels are rebuilt from it.

Overfitting controls:
- **Decision tree**: at most 3 levels (`max_depth=3`) and at least 5 orders per leaf (`min_samples_leaf=5`), so it cannot carve out single orders.
- **Logistic regression**: regularized (`C=1`).
- Both use `class_weight="balanced"` because positives are rare.

In [ ]:
strat_flag = (orders["purchase_revenue"] >= orders["purchase_revenue"].quantile(0.90)).astype(int)
idx_train, idx_test = train_test_split(orders.index, test_size=0.2, stratify=strat_flag,
                                       random_state=RANDOM_STATE)

hv_cut = float(orders.loc[idx_train, "purchase_revenue"].quantile(0.90))   # training only
y4o = (orders["purchase_revenue"] >= hv_cut).astype(int)
X4o_train, X4o_test = X_orders.loc[idx_train], X_orders.loc[idx_test]
y4o_train, y4o_test = y4o.loc[idx_train], y4o.loc[idx_test]
print(f"High-value cut-off (training P90): ${hv_cut:.2f}   (Table 6, all orders: $137.00)")
print(f"High-value orders  train: {y4o_train.sum()} of {len(y4o_train)}   test: {y4o_test.sum()} of {len(y4o_test)}")

tree_o = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, class_weight="balanced",
                                random_state=RANDOM_STATE)
logit_o = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000)
base_o = DummyClassifier(strategy="most_frequent")

p4o_test, p4o_cv, p4o_overfit = {}, {}, {}
cv_scoring = ["precision", "recall", "f1", "roc_auc"]
for name, model in [("Baseline (not high-value)", base_o), ("Decision tree", tree_o),
                    ("Logistic regression", logit_o)]:
    cv = cross_validate(model, X4o_train, y4o_train, cv=cv5, scoring=cv_scoring)
    p4o_cv[name] = {m: float(np.nanmean(cv[f"test_{m}"])) for m in cv_scoring}
    model.fit(X4o_train, y4o_train)
    p4o_test[name] = clf_scores(y4o_test, model.predict(X4o_test), model.predict_proba(X4o_test)[:, 1])
    p4o_overfit[name] = {"train_f1": f1_score(y4o_train, model.predict(X4o_train), zero_division=0),
                         "test_f1": p4o_test[name]["f1"]}

print("Test set:"); display(pd.DataFrame(p4o_test).T)
print("5-fold cross-validation on the training orders (steadier with so few positives):")
display(pd.DataFrame(p4o_cv).T)

**Interpretation of important features.**
- The **decision tree's splits read as plain rules**: each line is a yes/no question, and the leaves say "high-value" (class 1) or not (class 0).
- **Logistic regression coefficients** show the direction of each feature: positive raises the chance of a high-value order, negative lowers it.

In [ ]:
print(export_text(tree_o, feature_names=list(X_orders.columns)))
hv_coefs = (pd.DataFrame({"feature": X_orders.columns, "coefficient": logit_o.coef_[0]})
              .assign(abs_c=lambda d: d["coefficient"].abs())
              .sort_values("abs_c", ascending=False).drop(columns="abs_c"))
hv_coefs.head(10)

In [ ]:
p4o_overfit = pd.DataFrame(p4o_overfit).T
p4o_overfit["gap"] = p4o_overfit["train_f1"] - p4o_overfit["test_f1"]
p4o_overfit

### 8B. Quiet days

- **Cut-off**: P25 of the first 78 days only.
- **Features**: day of week, holiday flag, and `lag7` (events on the same weekday one week earlier, in thousands so the logistic coefficient is readable). The first 7 days have no `lag7` and are dropped from training.
- **Split**: time-based; the last 14 days are the test set, as in the forecast.

**Caution from the plan:** the 14 test days hold only about 3 quiet days, so one right or wrong day changes recall by about 33 points. Treat these results as an early estimate.

In [ ]:
quiet_cut = float(train_days["events"].quantile(0.25))   # training only
days["quiet"] = (days["events"] < quiet_cut).astype(int)
days["lag7_k"] = days["lag7"] / 1000

X_days = pd.get_dummies(days[["day_of_week"]], drop_first=True, dtype=float)
X_days["holiday"] = days["holiday"]
X_days["lag7_k"] = days["lag7_k"]

train_mask = np.arange(len(days)) < N_TRAIN_DAYS
X4d_train = X_days[train_mask].dropna();  y4d_train = days.loc[X4d_train.index, "quiet"]
X4d_test = X_days[~train_mask];           y4d_test = days.loc[X4d_test.index, "quiet"]
print(f"Quiet-day cut-off (training P25): {quiet_cut:,.0f} events   (Table 6, all days: 754)")
print(f"Quiet days  train: {y4d_train.sum()} of {len(y4d_train)}   test: {y4d_test.sum()} of {len(y4d_test)}")

tree_d = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, class_weight="balanced",
                                random_state=RANDOM_STATE)
logit_d = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000)
base_d = DummyClassifier(strategy="most_frequent")

p4d_test, p4d_overfit, day_flags = {}, {}, {}
for name, model in [("Baseline (not quiet)", base_d), ("Decision tree", tree_d),
                    ("Logistic regression", logit_d)]:
    model.fit(X4d_train, y4d_train)
    pred = model.predict(X4d_test)
    day_flags[name] = pred
    p4d_test[name] = clf_scores(y4d_test, pred)
    p4d_overfit[name] = {"train_f1": f1_score(y4d_train, model.predict(X4d_train), zero_division=0),
                         "test_f1": p4d_test[name]["f1"]}
p4d_table = pd.DataFrame(p4d_test).T.drop(columns="auc")
p4d_table

In [ ]:
print(export_text(tree_d, feature_names=list(X_days.columns)))
quiet_coefs = (pd.DataFrame({"feature": X_days.columns, "coefficient": logit_d.coef_[0]})
                 .sort_values("coefficient", key=np.abs, ascending=False))
display(quiet_coefs)

p4d_overfit = pd.DataFrame(p4d_overfit).T
p4d_overfit["gap"] = p4d_overfit["train_f1"] - p4d_overfit["test_f1"]
display(p4d_overfit)

flags_table = pd.DataFrame({"date": X4d_test.index.strftime("%Y-%m-%d"),
                            "weekday": days.loc[X4d_test.index, "day_of_week"].values,
                            "events": days.loc[X4d_test.index, "events"].values,
                            "actual_quiet": y4d_test.values,
                            "tree_flag": day_flags["Decision tree"],
                            "logit_flag": day_flags["Logistic regression"]})
flags_table

In [ ]:
RESULTS["p4"] = {
    "orders": {
        "cutoff": hv_cut,
        "n_pos_train": int(y4o_train.sum()), "n_pos_test": int(y4o_test.sum()),
        "metrics": pd.DataFrame(p4o_test).T.reset_index(names="model").to_dict("records"),
        "cv": pd.DataFrame(p4o_cv).T.reset_index(names="model").to_dict("records"),
        "overfit": p4o_overfit.reset_index(names="model").to_dict("records"),
        "tree_rules": export_text(tree_o, feature_names=list(X_orders.columns)),
    },
    "days": {
        "cutoff": quiet_cut,
        "n_pos_train": int(y4d_train.sum()), "n_pos_test": int(y4d_test.sum()),
        "metrics": p4d_table.reset_index(names="model").to_dict("records"),
        "overfit": p4d_overfit.reset_index(names="model").to_dict("records"),
        "tree_rules": export_text(tree_d, feature_names=list(X_days.columns)),
        "flags": flags_table.to_dict("records"),
    },
}

---
## 9. Summary: every model against its baseline

The rule in Section 5 of the plan: **a model is kept only if it beats its baseline on the test set**, and when two models score about the same, the simpler one is kept.

Each task is judged on its main metric:

| Task | Main metric | Better when |
|---|---|---|
| #1 Buyer | AUC | higher |
| #2 Order value / daily traffic | MAE | lower |
| #3 Order / traffic range | distance of coverage from 80% | lower |
| #4 High-value order / quiet day | Recall (F1 shown too) | higher |

In [ ]:
def row(problem, task, model, metric, value, baseline, higher_is_better):
    beats = value > baseline if higher_is_better else value < baseline
    return {"problem": problem, "task": task, "model": model, "metric": metric,
            "model_value": value, "baseline_value": baseline, "beats_baseline": bool(beats)}

b1 = p1_scores["Baseline (no buyer)"]
b2o = p2o_scores["Baseline (training median)"]
b2d = p2d_scores["Baseline (seasonal naive)"]
gap80 = lambda c: abs(c - 0.80)
b3o = p3o_scores["Baseline (fixed range)"]
b3d = p3d_scores["Baseline (last week ± typical change)"]
b4o = p4o_test["Baseline (not high-value)"]
b4d = p4d_test["Baseline (not quiet)"]

summary = pd.DataFrame([
    row("#1", "Buyer", "Logistic regression", "AUC", p1_scores["Logistic regression"]["auc"], b1["auc"], True),
    row("#1", "Buyer", "Gradient boosting", "AUC", p1_scores["Gradient boosting"]["auc"], b1["auc"], True),
    row("#2", "Order value", "Ridge regression", "MAE ($)", p2o_scores["Ridge regression"]["MAE"], b2o["MAE"], False),
    row("#2", "Daily traffic", "Holt-Winters", "MAE (events)", p2d_scores["Holt-Winters"]["MAE"], b2d["MAE"], False),
    row("#3", "Order range", "Quantile regression", "|coverage − 80%|",
        gap80(p3o_scores["Quantile regression"]["coverage"]), gap80(b3o["coverage"]), False),
    row("#3", "Traffic range", "Holt-Winters interval", "|coverage − 80%|",
        gap80(p3d_scores["Holt-Winters 80% interval"]["coverage"]), gap80(b3d["coverage"]), False),
    row("#4", "High-value order", "Decision tree", "Recall", p4o_test["Decision tree"]["recall"], b4o["recall"], True),
    row("#4", "High-value order", "Logistic regression", "Recall", p4o_test["Logistic regression"]["recall"], b4o["recall"], True),
    row("#4", "Quiet day", "Decision tree", "Recall", p4d_test["Decision tree"]["recall"], b4d["recall"], True),
    row("#4", "Quiet day", "Logistic regression", "Recall", p4d_test["Logistic regression"]["recall"], b4d["recall"], True),
])
summary

### Overfitting summary (Section 5.5)

All training-vs-test comparisons in one place. The data are small (127 buyers, 123 orders, 92 days), so some gap is normal; a **large** gap is the warning sign.

How to read the `train − test` column:
- For **scores** (AUC, F1, coverage), a large **positive** value is the warning sign: the model looked better on data it had already seen.
- For **errors** (RMSE, MAE), a large **negative** value is the warning sign: the model missed by more on new data.
- A value with the "good" sign (for example, a lower test RMSE) usually just means the test set happened to have fewer extreme values, such as the $551 order landing in the training set. Each model was also protected by regularization or a depth limit, and cross-validation was used where the data allowed.

In [ ]:
overfit_summary = pd.DataFrame([
    {"task": "#1 Buyer", "model": m, "metric": "AUC", "train": r["train_auc"], "test": r["test_auc"]}
    for m, r in p1_overfit.iterrows()] + [
    {"task": "#2 Order value", "model": "Ridge regression", "metric": "RMSE ($)",
     "train": p2o_overfit.iloc[0]["train_RMSE"], "test": p2o_overfit.iloc[0]["test_RMSE"]},
    {"task": "#2 Daily traffic", "model": "Holt-Winters", "metric": "MAE (events)",
     "train": p2d_overfit.loc["Holt-Winters", "train_MAE"], "test": p2d_overfit.loc["Holt-Winters", "test_MAE"]},
    {"task": "#3 Order range", "model": "Quantile regression", "metric": "Coverage",
     "train": p3o_overfit.iloc[0]["train_coverage"], "test": p3o_overfit.iloc[0]["test_coverage"]},
    {"task": "#3 Traffic range", "model": "Holt-Winters interval", "metric": "Coverage",
     "train": p3d_overfit.iloc[0]["train_coverage"], "test": p3d_overfit.iloc[0]["test_coverage"]}] + [
    {"task": "#4 High-value order", "model": m, "metric": "F1", "train": r["train_f1"], "test": r["test_f1"]}
    for m, r in p4o_overfit.iterrows() if not m.startswith("Baseline")] + [
    {"task": "#4 Quiet day", "model": m, "metric": "F1", "train": r["train_f1"], "test": r["test_f1"]}
    for m, r in p4d_overfit.iterrows() if not m.startswith("Baseline")])
overfit_summary["train − test"] = overfit_summary["train"] - overfit_summary["test"]
overfit_summary

---
## 10. Export `predictions.json` for the dashboard

Everything collected in `RESULTS` is written to **`predictions.json`**. In Colab the file downloads automatically.

**Next step:** copy `predictions.json` into the `ecommerce-gms-main/` folder (next to `data.json`), start the dashboard (`python3 -m http.server 8000`) and open the **Predictions** panel.

In [ ]:
def clean(o):
    # Converts numpy/pandas values to plain Python so json can write them; NaN -> null
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.strftime("%Y-%m-%d")
    return o

RESULTS["summary"] = summary.to_dict("records")
RESULTS["overfit_summary"] = overfit_summary.to_dict("records")
RESULTS["meta"] = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "source_csv": os.path.basename(CSV_PATH),
    "random_state": RANDOM_STATE,
    "n_rows": int(len(df)), "n_visitors": int(df["user_pseudo_id"].nunique()),
    "n_orders": int(len(orders)), "n_days": int(len(days)),
    "train_days": [train_days.index[0].strftime("%Y-%m-%d"), train_days.index[-1].strftime("%Y-%m-%d")],
    "test_days": [test_days.index[0].strftime("%Y-%m-%d"), test_days.index[-1].strftime("%Y-%m-%d")],
}

with open("predictions.json", "w", encoding="utf-8") as f:
    json.dump(clean(RESULTS), f, indent=1)
print(f"Wrote predictions.json ({os.path.getsize('predictions.json') / 1024:.0f} KB)")

if IN_COLAB:
    colab_files.download("predictions.json")